# Notebook 20 — Fine-tuning di GPT via API

**Modulo:** 20 — Fine-tuning di GPT via API
**Tempo stimato:** 1,5-2 ore (la parte di preparazione dati è interamente
eseguibile gratis; avviare un job reale è opzionale e a pagamento)

## Cosa costruirai

Al termine di questo notebook avrai:
- Un mini-dataset italiano (26 esempi) in formato conversazionale corretto
  per il fine-tuning via API (Modulo 20, § 2), diviso in training/validation
  e salvato come `.jsonl`
- Una stima del numero di token e del costo del job, prima di caricare
  nulla (Modulo 20, § 3)
- Il codice completo — protetto da un flag esplicito — per caricare i file,
  avviare il job, fare polling dello stato e recuperare il modello risultante
  (Modulo 20, § 4-5)
- Un esempio di dataset per il caso pratico "function calling più
  affidabile" (Modulo 20, § 6)
- Una funzione di pipeline riutilizzabile per preparare dataset futuri

## Prerequisiti

```bash
pip install openai python-dotenv tiktoken
```

Serve un file `.env` nella cartella del progetto con:
```
OPENAI_API_KEY=sk-...
```

> **Attenzione:** questo notebook interagisce con un servizio a pagamento.
> La preparazione del dataset (celle 1-5) è gratuita e non richiede una
> chiave valida per essere capita, ma **richiede** una chiave reale per le
> chiamate di inferenza di prova. L'avvio effettivo di un job di
> fine-tuning è protetto dal flag `ESEGUI_JOB_REALE`, spento di default: **non
> lanciarlo per curiosità**, ha un costo reale (Modulo 20, § 3-4).

---
## Cella 0 — Setup

In [ ]:
import os
import json
import random
import time
from dotenv import load_dotenv

load_dotenv()

OPENAI_KEY = os.getenv("OPENAI_API_KEY")

def verifica_chiave(nome: str, chiave: str | None) -> None:
    if chiave:
        print(f"OK {nome}: {chiave[:8]}...")
    else:
        print(f"MANCANTE {nome}: controlla il tuo file .env")

verifica_chiave("OPENAI_API_KEY", OPENAI_KEY)

from openai import OpenAI

cliente_openai = OpenAI(api_key=OPENAI_KEY)
random.seed(42)

---
## Cella 1 — Il caso pratico

Modulo 20, § 2: un giornale digitale vuole generare automaticamente, per
ogni articolo, tre campi in un formato JSON fisso: **titolo**, **parole
chiave SEO** e **riassunto**. Lo stile deve essere quello editoriale del
giornale, non uno stile generico — è per questo che serve fine-tuning,
non solo un prompt di sistema (che pure funzionerebbe, ma in modo meno
consistente su larga scala).

Definiamo il compito con un unico `SYSTEM_PROMPT` fisso, e poi 26 coppie
articolo → output atteso, scritte a mano.

In [ ]:
SYSTEM_PROMPT = (
    "Dato un articolo di cronaca, restituisci un dizionario JSON con i "
    "campi 'titolo', 'seo' (lista di 3-5 parole chiave) e 'riassunto' "
    "(una frase)."
)

# Ogni tupla: (articolo, titolo, parole_seo, riassunto)
ARTICOLI_GREZZI = [
    ("Il Comune ha approvato ieri il nuovo piano per la mobilità urbana, che prevede piste ciclabili aggiuntive e un potenziamento del trasporto pubblico entro il 2027.",
     "Il Comune approva il nuovo piano per la mobilità urbana",
     ["mobilità urbana", "piano comunale", "piste ciclabili", "trasporto pubblico"],
     "Il Comune ha approvato un piano per la mobilità urbana con nuove piste ciclabili e trasporto pubblico potenziato."),
    ("La squadra locale di calcio ha conquistato la promozione in serie superiore dopo una stagione da record, con soli due punti persi in tutto il campionato.",
     "Promozione storica per la squadra locale di calcio",
     ["calcio", "promozione", "campionato", "sport locale"],
     "La squadra locale è stata promossa dopo una stagione quasi perfetta."),
    ("Un gruppo di ricercatori dell'università ha sviluppato un nuovo materiale biodegradabile a base di alghe, pensato per sostituire alcune plastiche monouso.",
     "Un nuovo materiale a base di alghe per sostituire la plastica",
     ["biodegradabile", "alghe", "plastica monouso", "ricerca universitaria"],
     "Alcuni ricercatori hanno creato un materiale biodegradabile a base di alghe per sostituire la plastica."),
    ("La biblioteca civica ha esteso l'orario di apertura serale su richiesta degli studenti universitari, che potranno ora studiare fino a mezzanotte durante la sessione d'esami.",
     "La biblioteca civica apre fino a mezzanotte per gli esami",
     ["biblioteca civica", "orari", "studenti universitari", "sessione esami"],
     "La biblioteca civica ha esteso l'orario serale per gli studenti durante gli esami."),
    ("Un'azienda locale ha lanciato un servizio di consegna a domicilio tramite piccoli robot autonomi, attivo per ora solo nel centro storico della città.",
     "Robot autonomi per le consegne nel centro storico",
     ["robot autonomi", "consegna a domicilio", "centro storico", "innovazione"],
     "Un'azienda locale ha avviato consegne con robot autonomi nel centro storico."),
    ("Il mercato settimanale di prodotti a chilometro zero ha registrato quest'anno un aumento del trenta percento di visitatori rispetto alla stagione precedente.",
     "Boom di visitatori per il mercato a chilometro zero",
     ["mercato locale", "chilometro zero", "agricoltura", "visitatori"],
     "Il mercato a chilometro zero ha registrato un aumento del trenta percento di visitatori."),
    ("Una nuova pista ciclabile collegera il centro città alla zona industriale, riducendo secondo le stime comunali il traffico veicolare nelle ore di punta.",
     "Nuova pista ciclabile tra centro e zona industriale",
     ["pista ciclabile", "traffico", "zona industriale", "mobilità sostenibile"],
     "Una nuova pista ciclabile collegera centro e zona industriale riducendo il traffico."),
    ("Il teatro comunale ha annunciato la nuova stagione, con dieci spettacoli tra prosa e danza contemporanea in programma da ottobre a maggio.",
     "Annunciata la nuova stagione del teatro comunale",
     ["teatro comunale", "stagione teatrale", "danza contemporanea", "prosa"],
     "Il teatro comunale ha annunciato dieci spettacoli per la nuova stagione."),
    ("I dati dell'osservatorio economico locale mostrano una crescita dell'occupazione nel settore tecnologico, con oltre duecento nuove assunzioni negli ultimi sei mesi.",
     "Cresce l'occupazione nel settore tecnologico locale",
     ["occupazione", "settore tecnologico", "assunzioni", "economia locale"],
     "Il settore tecnologico locale ha registrato oltre duecento nuove assunzioni in sei mesi."),
    ("La scuola elementare del quartiere ha inaugurato un nuovo laboratorio di robotica educativa, finanziato con fondi europei destinati alla digitalizzazione scolastica.",
     "Nuovo laboratorio di robotica nella scuola del quartiere",
     ["robotica educativa", "scuola elementare", "fondi europei", "digitalizzazione"],
     "La scuola del quartiere ha aperto un laboratorio di robotica grazie a fondi europei."),
    ("Il festival cittadino di musica jazz tornera quest'estate con dieci concerti gratuiti in altrettante piazze del centro storico.",
     "Il festival jazz torna con dieci concerti gratuiti",
     ["festival jazz", "concerti gratuiti", "centro storico", "musica"],
     "Il festival jazz cittadino tornera con dieci concerti gratuiti in centro."),
    ("L'ospedale cittadino ha attivato un nuovo servizio di telemedicina per i pazienti cronici, che potranno prenotare visite di controllo online.",
     "Telemedicina per i pazienti cronici in ospedale",
     ["telemedicina", "ospedale", "pazienti cronici", "sanità digitale"],
     "L'ospedale ha attivato un servizio di telemedicina per i pazienti cronici."),
    ("Il comune ha stanziato fondi per la ristrutturazione del parco pubblico più grande della città, con lavori che inizieranno la prossima primavera.",
     "Fondi comunali per ristrutturare il parco più grande",
     ["parco pubblico", "ristrutturazione", "fondi comunali", "lavori pubblici"],
     "Il Comune ristrutturera il parco pubblico più grande con lavori dalla prossima primavera."),
    ("Una start-up locale ha sviluppato un'app per ridurre lo spreco alimentare nei ristoranti, mettendo in contatto locali con eccedenze e associazioni benefiche.",
     "Un'app locale contro lo spreco alimentare nei ristoranti",
     ["spreco alimentare", "start-up", "app", "associazioni benefiche"],
     "Una start-up locale ha creato un'app per collegare ristoranti con eccedenze alle associazioni benefiche."),
    ("Il numero di iscritti ai corsi serali di lingua straniera organizzati dal Comune e raddoppiato nell'ultimo anno, spingendo l'amministrazione ad aprire nuove classi.",
     "Raddoppiano gli iscritti ai corsi serali di lingue",
     ["corsi serali", "lingue straniere", "Comune", "formazione adulti"],
     "Gli iscritti ai corsi serali di lingue del Comune sono raddoppiati in un anno."),
    ("La squadra di pallacanestro femminile ha raggiunto la finale del torneo regionale per la prima volta nella storia del club.",
     "Storica finale regionale per la pallacanestro femminile",
     ["pallacanestro femminile", "torneo regionale", "finale", "sport locale"],
     "La squadra di pallacanestro femminile ha raggiunto per la prima volta la finale regionale."),
    ("Il centro storico sara chiuso al traffico nei fine settimana estivi per favorire la fruizione pedonale di bar e ristoranti.",
     "Centro storico pedonale nei weekend estivi",
     ["centro storico", "zona pedonale", "traffico", "estate"],
     "Il centro storico sara chiuso al traffico nei weekend estivi."),
    ("Un progetto pilota di orti urbani condivisi ha coinvolto oltre cento famiglie in tre quartieri periferici della città.",
     "Oltre cento famiglie negli orti urbani condivisi",
     ["orti urbani", "progetto pilota", "quartieri periferici", "comunità"],
     "Oltre cento famiglie hanno partecipato al progetto di orti urbani condivisi."),
    ("La stazione ferroviaria ha completato i lavori di accessibilità, con nuovi ascensori e percorsi tattili per persone con disabilità visiva.",
     "Nuova accessibilità per la stazione ferroviaria cittadina",
     ["accessibilità", "stazione ferroviaria", "disabilità", "lavori pubblici"],
     "La stazione ferroviaria ha completato i lavori per renderla più accessibile."),
    ("Il museo civico ha digitalizzato l'intera collezione di reperti archeologici, rendendola consultabile gratuitamente online da tutto il mondo.",
     "Il museo civico digitalizza la collezione archeologica",
     ["museo civico", "digitalizzazione", "archeologia", "accesso online"],
     "Il museo civico ha reso consultabile online l'intera collezione archeologica."),
    ("Un'ondata di caldo anomalo ha spinto il Comune ad aprire centri climatizzati gratuiti per gli anziani nei quartieri più colpiti.",
     "Centri climatizzati gratuiti per gli anziani durante il caldo",
     ["ondata di caldo", "anziani", "centri climatizzati", "emergenza caldo"],
     "Il Comune ha aperto centri climatizzati gratuiti per gli anziani durante l'ondata di caldo."),
    ("La cooperativa sociale del quartiere ha avviato un programma di riparazione gratuita di piccoli elettrodomestici, per ridurre i rifiuti elettronici.",
     "Riparazioni gratuite di elettrodomestici contro i rifiuti elettronici",
     ["riparazione", "elettrodomestici", "rifiuti elettronici", "cooperativa sociale"],
     "Una cooperativa sociale ripara gratuitamente piccoli elettrodomestici per ridurre i rifiuti elettronici."),
    ("Il liceo scientifico cittadino ha vinto la fase nazionale di una competizione di robotica, superando oltre cento scuole in tutta Italia.",
     "Il liceo scientifico vince la finale nazionale di robotica",
     ["liceo scientifico", "robotica", "competizione nazionale", "studenti"],
     "Il liceo scientifico ha vinto la finale nazionale di una competizione di robotica."),
    ("Il porto turistico ha inaugurato una nuova banchina dedicata alle imbarcazioni a propulsione elettrica, prima infrastruttura del genere nella regione.",
     "Nuova banchina per barche elettriche nel porto turistico",
     ["porto turistico", "imbarcazioni elettriche", "infrastruttura", "sostenibilità"],
     "Il porto turistico ha inaugurato la prima banchina regionale per barche elettriche."),
    ("Un'associazione di quartiere ha organizzato una raccolta fondi per restaurare la fontana storica della piazza principale, danneggiata da un fulmine.",
     "Raccolta fondi per restaurare la fontana storica",
     ["fontana storica", "raccolta fondi", "restauro", "associazione di quartiere"],
     "Un'associazione ha avviato una raccolta fondi per restaurare la fontana storica danneggiata."),
    ("Il servizio di bike sharing comunale ha superato le centomila corse nel primo anno di attività, superando le previsioni iniziali dell'amministrazione.",
     "Il bike sharing comunale supera centomila corse nel primo anno",
     ["bike sharing", "mobilità condivisa", "Comune", "primo anno"],
     "Il bike sharing comunale ha superato centomila corse nel suo primo anno."),
]

print(f"Numero totale di articoli grezzi: {len(ARTICOLI_GREZZI)}")

---
## Cella 2 — Costruzione del formato a messaggi

Modulo 20, § 2: ogni esempio di training è un oggetto con una lista
`messages` (system, user, assistant), esattamente come una chiamata normale
all'API (Modulo 01, Modulo 10). Il messaggio `assistant` contiene il
**completion atteso** — qui, la stringa JSON con `titolo`, `seo`,
`riassunto`.

In [ ]:
def costruisci_esempio(articolo: str, titolo: str, seo: list[str], riassunto: str) -> dict:
    """Costruisce un esempio nel formato a messaggi per il fine-tuning."""
    output_atteso = json.dumps(
        {"titolo": titolo, "seo": seo, "riassunto": riassunto},
        ensure_ascii=False,
    )
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": articolo},
            {"role": "assistant", "content": output_atteso},
        ]
    }


ESEMPI = [costruisci_esempio(*a) for a in ARTICOLI_GREZZI]

print("Esempio di esempio (indice 0):")
print(json.dumps(ESEMPI[0], ensure_ascii=False, indent=2))

---
## Cella 3 — Split train/validation e scrittura dei file JSONL

Ogni riga del file `.jsonl` è un esempio indipendente. Come per il
fine-tuning con Transformers (Modulo 18, § 3), dividiamo in training e
validation per poter monitorare la qualità durante l'addestramento invece
di scoprirla solo alla fine.

In [ ]:
def scrivi_jsonl(esempi: list[dict], percorso: str) -> None:
    with open(percorso, "w", encoding="utf-8") as f:
        for esempio in esempi:
            f.write(json.dumps(esempio, ensure_ascii=False) + "\n")


indici = list(range(len(ESEMPI)))
random.shuffle(indici)

N_VALIDATION = 6
indici_validation = set(indici[:N_VALIDATION])

esempi_train = [e for i, e in enumerate(ESEMPI) if i not in indici_validation]
esempi_validation = [e for i, e in enumerate(ESEMPI) if i in indici_validation]

PATH_TRAIN = "finetuning_train.jsonl"
PATH_VALIDATION = "finetuning_validation.jsonl"

scrivi_jsonl(esempi_train, PATH_TRAIN)
scrivi_jsonl(esempi_validation, PATH_VALIDATION)

print(f"Training:   {len(esempi_train)} esempi -> {PATH_TRAIN}")
print(f"Validation: {len(esempi_validation)} esempi -> {PATH_VALIDATION}")

---
## Cella 4 — Quanti token stiamo per addestrare?

Modulo 20, § 3: `tokens_addestrati` è la somma dei token di **tutto** il
dataset (system + user + assistant, su ogni esempio), non solo dei
completion. Contiamo prima di caricare nulla, con `tiktoken` (già visto
nel Modulo 01/08 per stimare i costi di inferenza).

In [ ]:
import tiktoken

ENCODING = tiktoken.get_encoding("o200k_base")  # verifica l'encoding aggiornato nella doc ufficiale se cambia

def conta_token_esempio(esempio: dict) -> int:
    """Somma i token di tutti i messaggi di un esempio (system + user + assistant)."""
    testo_completo = "\n".join(m["content"] for m in esempio["messages"])
    return len(ENCODING.encode(testo_completo))

def conta_token_dataset(esempi: list[dict]) -> int:
    return sum(conta_token_esempio(e) for e in esempi)


token_train = conta_token_dataset(esempi_train)
token_validation = conta_token_dataset(esempi_validation)

print(f"Token nel training set:   {token_train}")
print(f"Token nel validation set: {token_validation}")
print(f"Token totali:             {token_train + token_validation}")

---
## Cella 5 — Stima del costo del job

Modulo 20, § 3:

```
costo = tokens_addestrati × numero_di_epoch × prezzo_per_token_di_addestramento
```

Il prezzo di addestramento è **diverso** dal prezzo di inferenza dello
stesso modello, e tipicamente più alto. Definiamo una funzione dedicata,
distinta da `calcola_costo` (Modulo 08) — quella resta per stimare il
costo delle chiamate di **inferenza**, questa stima il costo
dell'**addestramento**. Solo il conteggio dei token di training entra in
questa formula: il `validation_file` viene usato per calcolare metriche
durante il training, non viene addestrato sopra, quindi non conta nel
costo.

In [ ]:
# Prezzo di esempio illustrativo, in USD per milione di token di addestramento.
# Verifica sempre i prezzi aggiornati sul sito del provider prima di stimare un budget reale.
PREZZI_ADDESTRAMENTO = {
    "gpt-5-mini": 8.00,
}

def calcola_costo_addestramento(token_totali: int, num_epoch: int, modello: str) -> dict:
    """Stima il costo di un job di fine-tuning (Modulo 20, § 3)."""
    if modello not in PREZZI_ADDESTRAMENTO:
        raise ValueError(f"Modello '{modello}' non presente in PREZZI_ADDESTRAMENTO.")

    prezzo_per_milione = PREZZI_ADDESTRAMENTO[modello]
    token_fatturati = token_totali * num_epoch
    costo_usd = (token_fatturati / 1_000_000) * prezzo_per_milione

    return {
        "modello": modello,
        "token_dataset": token_totali,
        "num_epoch": num_epoch,
        "token_fatturati": token_fatturati,
        "costo_stimato_usd": round(costo_usd, 4),
    }


stima = calcola_costo_addestramento(token_train, num_epoch=3, modello="gpt-5-mini")
for k, v in stima.items():
    print(f"  {k}: {v}")

---
## Cella 6 — Attenzione: da qui in poi si interagisce con l'API reale

Modulo 20, § 4: caricare i file e avviare il job **non è gratuito** e
**non è istantaneo**. Il flag `ESEGUI_JOB_REALE` protegge tutto il resto
del notebook: resta a `False` finché non hai deciso consapevolmente di
spendere e aspettare.

In [ ]:
ESEGUI_JOB_REALE = False  # cambia a True SOLO se vuoi davvero avviare un job a pagamento

if ESEGUI_JOB_REALE:
    file_training = cliente_openai.files.create(
        file=open(PATH_TRAIN, "rb"),
        purpose="fine-tune",
    )
    file_validation = cliente_openai.files.create(
        file=open(PATH_VALIDATION, "rb"),
        purpose="fine-tune",
    )
    print("File di training caricato:   ", file_training.id)
    print("File di validation caricato:  ", file_validation.id)
else:
    print("ESEGUI_JOB_REALE è False: nessun file caricato, nessun costo sostenuto.")
    file_training = None
    file_validation = None

In [ ]:
if ESEGUI_JOB_REALE:
    job = cliente_openai.fine_tuning.jobs.create(
        training_file=file_training.id,
        validation_file=file_validation.id,
        model="gpt-5-mini-2026-01-01",  # verifica l'ID modello fine-tunabile aggiornato nella doc ufficiale
        hyperparameters={"n_epochs": 3},
    )
    print("Job creato:", job.id)
    print("Stato iniziale:", job.status)
else:
    print("ESEGUI_JOB_REALE è False: nessun job creato.")
    job = None

---
## Cella 7 — Monitorare il job

Modulo 20, § 4, passo 3: il job è **asincrono**. Non blocchi il tuo
programma in attesa di una risposta sincrona: interroghi periodicamente lo
stato (*polling*) finché non è `succeeded`, `failed` o `cancelled`. Queste
funzioni sono definite e sicure da eseguire (non costano nulla in sé): il
costo è già stato generato quando il job è stato creato nella Cella 6, non
quando lo controlli.

In [ ]:
def attendi_completamento_job(job_id: str, intervallo_secondi: int = 30) -> object:
    """Fa polling dello stato del job finché non termina (successo, errore o cancellazione)."""
    stati_finali = {"succeeded", "failed", "cancelled"}
    while True:
        stato_job = cliente_openai.fine_tuning.jobs.retrieve(job_id)
        print(f"[{time.strftime('%H:%M:%S')}] stato: {stato_job.status}")
        if stato_job.status in stati_finali:
            return stato_job
        time.sleep(intervallo_secondi)


def mostra_eventi_job(job_id: str, limite: int = 10) -> None:
    """Stampa gli ultimi eventi del job (utile per capire cosa sta succedendo)."""
    eventi = cliente_openai.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=limite)
    for evento in eventi.data:
        print(f"  {evento.created_at} — {evento.message}")


if ESEGUI_JOB_REALE and job is not None:
    stato_finale = attendi_completamento_job(job.id, intervallo_secondi=30)
    mostra_eventi_job(job.id)
else:
    print("Job non avviato in questa esecuzione: nulla da monitorare.")

---
## Cella 8 — Recuperare e usare il modello fine-tuned

Modulo 20, § 5: una volta pronto, il modello fine-tuned si usa **esattamente
come un modello normale** — stesso endpoint, stesso formato, cambia solo
`model`. Definiamo la funzione in modo generico e la proviamo intanto sul
modello **base** (una chiamata di inferenza normale, con un costo minimo
reale): quando il tuo job sarà pronto, ti basterà sostituire
`MODELLO_DA_USARE` con `job.fine_tuned_model`.

In [ ]:
def genera_output_articolo(articolo: str, modello: str) -> str:
    """Chiama il modello (base o fine-tuned) con lo stesso SYSTEM_PROMPT del dataset."""
    risposta = cliente_openai.chat.completions.create(
        model=modello,
        temperature=0,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": articolo},
        ],
    )
    return risposta.choices[0].message.content


# Se il job è stato completato con successo, usa il modello risultante.
# Altrimenti, dimostriamo il pattern con il modello base (costo minimo, reale).
if ESEGUI_JOB_REALE and job is not None and getattr(job, "fine_tuned_model", None):
    MODELLO_DA_USARE = job.fine_tuned_model
else:
    MODELLO_DA_USARE = "gpt-5-mini"

articolo_di_prova = (
    "Il Comune ha inaugurato oggi una nuova area giochi inclusiva nel "
    "parco cittadino, pensata anche per bambini con disabilità motorie."
)

output = genera_output_articolo(articolo_di_prova, MODELLO_DA_USARE)
print(f"Modello usato: {MODELLO_DA_USARE}")
print("Output:", output)

---
## Cella 9 — Caso pratico aggiuntivo: fine-tuning per function calling più affidabile

Modulo 20, § 6: un bot aziendale deve interrogare un sistema interno
"Client Database", ma il modello base a volte lo confonde con il nome di
una persona, o accetta un nome parziale invece di richiedere sempre il
nome completo. Un esempio di training per questo scenario usa il formato
**`tools`** — lo stesso già visto nel Modulo 11, non il vecchio formato
`functions`/`function_call` (deprecato) usato nel materiale originale —
con l'assistente che produce una `tool_call` solo quando ha tutte le
informazioni richieste.

In [ ]:
STRUMENTO_CLIENT_DB = {
    "type": "function",
    "function": {
        "name": "client_database",
        "description": "Restituisce le informazioni di un cliente dal sistema interno Client Database.",
        "parameters": {
            "type": "object",
            "properties": {
                "nome_completo": {
                    "type": "string",
                    "description": "Nome e cognome completi del cliente, es. 'Marco Rossi'",
                }
            },
            "required": ["nome_completo"],
        },
    },
}

# Esempio 1: l'utente fornisce solo il nome di battesimo -> il modello DEVE chiedere il cognome,
# non chiamare la funzione con dati incompleti.
esempio_funzioni_incompleto = {
    "messages": [
        {"role": "system", "content": "Sei un assistente interno. Usa client_database solo con il nome completo del cliente."},
        {"role": "user", "content": "Puoi cercarmi le informazioni di Marco su Client Database?"},
        {"role": "assistant", "content": "Mi serve il cognome completo di Marco per interrogare Client Database. Puoi indicarmelo?"},
    ],
    "tools": [STRUMENTO_CLIENT_DB],
}

# Esempio 2: l'utente fornisce il nome completo -> il modello chiama correttamente lo strumento.
esempio_funzioni_completo = {
    "messages": [
        {"role": "system", "content": "Sei un assistente interno. Usa client_database solo con il nome completo del cliente."},
        {"role": "user", "content": "Cerca le informazioni di Marco Rossi su Client Database."},
        {
            "role": "assistant",
            "content": None,
            "tool_calls": [
                {
                    "id": "call_1",
                    "type": "function",
                    "function": {"name": "client_database", "arguments": json.dumps({"nome_completo": "Marco Rossi"})},
                }
            ],
        },
    ],
    "tools": [STRUMENTO_CLIENT_DB],
}

print(json.dumps(esempio_funzioni_incompleto, ensure_ascii=False, indent=2))
print()
print(json.dumps(esempio_funzioni_completo, ensure_ascii=False, indent=2))

Un dataset reale per questo caso includerebbe decine di varianti di questi
due pattern (richieste con nome parziale, con nome completo, con "Client
Database" scritto in modi diversi...) — la tabella del Modulo 20, § 6
riassume il miglioramento atteso rispetto al modello base.

---
## Esercizio — completa il codice

**TODO:** aggiungi almeno 3 nuove tuple `(articolo, titolo, seo, riassunto)`
a `ARTICOLI_GREZZI`, ricostruisci `ESEMPI`, e ricalcola il numero totale di
token e il costo stimato del job con 3 epoch.

In [ ]:
# TODO: aggiungi almeno 3 nuove tuple a questa lista, sullo stesso schema di ARTICOLI_GREZZI
NUOVI_ARTICOLI = [
]

# TODO: costruisci esempi_estesi = ESEMPI + [costruisci_esempio(*a) for a in NUOVI_ARTICOLI]

# TODO: ricalcola il numero totale di token con conta_token_dataset() sul dataset esteso

# TODO: richiama calcola_costo_addestramento() con num_epoch=3 e stampa il risultato


---
## Sfida finale — pipeline completa di preparazione dataset

Costruisci una funzione `prepara_dataset_finetuning` che integra **tutti**
i passi di questo notebook (costruzione messaggi, split, scrittura JSONL,
conteggio token, stima costo), così da poterla riusare su qualunque nuovo
insieme di articoli, senza ripetere il codice a mano ogni volta.

In [ ]:
def prepara_dataset_finetuning(
    articoli_grezzi: list[tuple],
    system_prompt: str,
    path_train: str,
    path_validation: str,
    frazione_validation: float = 0.2,
    modello: str = "gpt-5-mini",
    num_epoch: int = 3,
) -> dict:
    """Pipeline completa: messaggi -> split -> file JSONL -> stima costo.

    Restituisce un dict con: n_train, n_validation, token_totali, stima_costo.
    """
    # TODO 1: costruisci la lista di esempi nel formato a messaggi, riusando
    # costruisci_esempio() (attenzione: qui system_prompt è un parametro,
    # non la variabile globale SYSTEM_PROMPT)

    # TODO 2: calcola n_validation = round(len(esempi) * frazione_validation),
    # poi mescola gli indici e dividi in esempi_train / esempi_validation
    # (stesso schema della Cella 3)

    # TODO 3: scrivi i due file JSONL con scrivi_jsonl()

    # TODO 4: calcola token_totali sul solo training set con conta_token_dataset()

    # TODO 5: chiama calcola_costo_addestramento(token_totali, num_epoch, modello)

    # TODO 6: restituisci un dict con n_train, n_validation, token_totali, stima_costo
    pass


# Prova la tua pipeline sul dataset originale, scrivendo in due nuovi file
risultato_pipeline = prepara_dataset_finetuning(
    ARTICOLI_GREZZI,
    SYSTEM_PROMPT,
    path_train="finetuning_train_v2.jsonl",
    path_validation="finetuning_validation_v2.jsonl",
    frazione_validation=0.2,
    num_epoch=3,
)

print(risultato_pipeline)

---
## Autovalutazione della settimana

Rispondi in questa cella prima di chiudere il notebook:

1. **Cosa ha funzionato bene questa settimana?**
   ...

2. **Cosa non hai capito del tutto?**
   ...

3. **Cosa cambieresti nel tuo approccio la prossima settimana?**
   ...

4. **Tempo reale dedicato:**
   ...

---
*Notebook 20 — Modulo 20 del Corso Completo di IA Generativa*